# Pre-processing a .CHA file for use in CE analysis

In [1]:
import os
import sys
import pandas as pd
from tqdm import tqdm

In [2]:
data_path = '../data'
chas_path = os.path.join(data_path, 'chas')
outputs_path = os.path.join(data_path, 'server_ready', 'cha_corpus.csv')

In [3]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

# Processing all CHA files

Note: the package used here was developed by Prof. Garber. Cite via:

Garber, L. (2019). CHA file python parser. Zenodo. https://doi.org/10.5281/zenodo.3364020

In [4]:
from shared.CHAFile import *

In [5]:
all_files = grab_all_files(chas_path)
# all_files

In [6]:
data = []
for f in all_files:
    chacha = ChaFile(f)
    meta_data_pieces = f.replace('.cha', '').split('/')
    for line in chacha.getLines():
        line['conversation_id'] = meta_data_pieces[-1]
        line['overlapping_text'] = bool(re.findall(r"(⌋|⌊|⌉|⌈)", line['text']))

        if meta_data_pieces[-2] in ['eng_n', 'eng_s']:
            line['corpus'] = meta_data_pieces[-3] + '-' + meta_data_pieces[-2]
        
        elif meta_data_pieces[-3] in ['instruction']:
            line['corpus'] = meta_data_pieces[-3] + '-' + meta_data_pieces[-2]
        
        else:
            line['corpus'] = meta_data_pieces[-2]
        
        data += [line.copy()]

data = pd.DataFrame(data)

In [7]:
data.head()

,document_line_no,utterance_no,speaker,text,bullet,recipient,conversation_id,overlapping_text,corpus,com,exp,mor,gra,gpx,act,pic,par,sit
0,10,1,A,Right .,"[200460, 200690]",ADULT,0638,False,callhome,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11,2,B,and I .,"[201140, 201610]",ADULT,0638,False,callhome,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12,3,B,I already got another &=yelling apartment .,"[201950, 203290]",ADULT,0638,False,callhome,there are background noises throughout this co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15,4,B,for when I move out &=bang .,"[203880, 204930]",ADULT,0638,False,callhome,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16,5,A,oh you did .,"[205210, 205610]",ADULT,0638,False,callhome,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Correcting utterances/removing CLAN specific formatting.

In [8]:
def corrected_text(text, contraction_replacement_nonce='CCOONNTTRRAACCTTIIOONN'):
    res = re.sub(r'\(\(.*?\)\)', ' ', str(text))
    # res = re.sub(r'\[.*?\]', ' ', res)
    
    # find contractions and preserve them . . .
    contractions = list(re.findall(r"\w+'\w+", res))
    for contraction in set(contractions):
        replacement = contraction.replace("'", contraction_replacement_nonce)
        res = res.replace(contraction, replacement)
    res = re.sub(r"(⌋|⌊|⌉|⌈)", '', res)
    res = res.replace(':', '')
    
    # remove numbers in parentheses (times???)
    res = re.sub(r'\(\d\.\d\)', ' ', res)
    
    # remove all other special characters.
    res = re.sub(r'[^\w\s]', ' ', res)
    
    res = re.sub(r'\s+', ' ', res).replace('[', ' ').replace(']', ' ').replace(contraction_replacement_nonce, "'")
    return res.strip()

In [9]:
data['raw_text'] = data['text'].values
data['text'] = [corrected_text(text) for text in tqdm(data['raw_text'].values)]

100%|██████████| 84432/84432 [00:00<00:00, 109058.91it/s]


In [10]:
data[['corpus', 'raw_text', 'text']].head(n=6)

,corpus,raw_text,text
0,callhome,Right .,Right
1,callhome,and I .,and I
2,callhome,I already got another &=yelling apartment .,I already got another yelling apartment
3,callhome,for when I move out &=bang .,for when I move out bang
4,callhome,oh you did .,oh you did
5,callhome,yeah .,yeah


## Create juxtaposed corpus: (x,y) pairs

In [11]:
max_turns_apart = 30

In [12]:
import warnings; warnings.filterwarnings("ignore")

corpus = []
for cid in tqdm(data['conversation_id'].unique()):
    sub = data.loc[data['conversation_id'].isin([cid])]
    sub_index = sub.index.values
    
    for i in sub_index:
        if i != sub_index[-1]:
            
            # speaker vs. other
            next_line_no = ( (sub_index > i) & (~sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]
            
            # speaker vs. self 
            next_line_no = ( (sub_index > i) & (sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]

100%|██████████| 243/243 [05:18<00:00,  1.31s/it]


In [13]:
data = pd.DataFrame(corpus)
data.head()

,document_line_no,utterance_no,speaker,text,bullet,recipient,conversation_id,overlapping_text,corpus,com,...,gpx,act,pic,par,sit,raw_text,next_speaker,next_text,next_utterance_no,next_utterance_delta_no
0,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,and I,2,0
1,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,I already got another yelling apartment,3,1
2,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,for when I move out bang,4,2
3,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,yeah,6,3
4,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,lipsmack it's on the corner of Columbia and Cole,8,4


In [14]:
rename_columns = dict()
for col in data.columns:
    if col == 'text':
        rename_columns[col] = 'x_utterance'
    elif col == 'next_text':
        rename_columns[col] = 'y_utterance'
    elif 'next_' in col:
        rename_columns[col]  = col.replace('next', 'y')
    else:
        rename_columns[col] = col

In [15]:
data = data.rename(columns=rename_columns)
data.head()

,document_line_no,utterance_no,speaker,x_utterance,bullet,recipient,conversation_id,overlapping_text,corpus,com,...,gpx,act,pic,par,sit,raw_text,y_speaker,y_utterance,y_utterance_no,y_utterance_delta_no
0,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,and I,2,0
1,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,I already got another yelling apartment,3,1
2,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,for when I move out bang,4,2
3,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,yeah,6,3
4,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,lipsmack it's on the corner of Columbia and Cole,8,4


In [19]:
data = data.rename(columns={'speaker':'x_speaker','utterance_no': 'x_turn_id', 'y_utterance_no': 'y_turn_id'})
data.head()

,document_line_no,x_turn_id,x_speaker,x_utterance,bullet,recipient,conversation_id,overlapping_text,corpus,com,...,gpx,act,pic,par,sit,raw_text,y_speaker,y_utterance,y_turn_id,y_utterance_delta_no
0,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,and I,2,0
1,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,I already got another yelling apartment,3,1
2,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,for when I move out bang,4,2
3,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,yeah,6,3
4,10,1,A,Right,"[200460, 200690]",ADULT,0638,False,callhome,NaN,...,NaN,NaN,NaN,NaN,NaN,Right .,B,lipsmack it's on the corner of Columbia and Cole,8,4


In [20]:
data['x_speaker'] = data['corpus'].astype(str) + '-' + data['conversation_id'].astype(str) + '-' + data['x_speaker'].astype(str)

data['y_speaker'] = data['corpus'].astype(str) + '-' + data['conversation_id'].astype(str) + '-' + data['y_speaker'].astype(str)

In [22]:
data['self'] = data['x_speaker'] == data['y_speaker']
data = data.sort_values(by=['corpus', 'conversation_id', 'x_turn_id', 'self', 'y_turn_id'])
data.index = range(len(data))

In [23]:
data[['corpus', 'x_utterance', 'y_utterance']].isna().mean()

corpus         0.0
x_utterance    0.0
y_utterance    0.0
dtype: float64

In [19]:
# corpus sanity check . . . make sure that conversation_ids are all unique 
k = data[['conversation_id', 'corpus']].drop_duplicates()
k['conversation_id'].nunique(), k['conversation_id'].nunique()/len(k)

(244, 1.0)

In [24]:
data[['corpus', 'x_utterance', 'x_speaker', 'y_speaker', 'y_utterance', 'x_turn_id', 'y_turn_id']].head()

,corpus,x_utterance,x_speaker,y_speaker,y_utterance,x_turn_id,y_turn_id
0,callfriend-eng_n,hhh hhh hhh hhh,callfriend-eng_n-4175-M2,callfriend-eng_n-4175-M1,whatt they didn't say that in the thing,1,2
1,callfriend-eng_n,hhh hhh hhh hhh,callfriend-eng_n-4175-M2,callfriend-eng_n-4175-M1,publically distributed,1,5
2,callfriend-eng_n,hhh hhh hhh hhh,callfriend-eng_n-4175-M2,callfriend-eng_n-4175-M1,I didn't read that,1,7
3,callfriend-eng_n,hhh hhh hhh hhh,callfriend-eng_n-4175-M2,callfriend-eng_n-4175-M1,hhh well the xxx was,1,9
4,callfriend-eng_n,hhh hhh hhh hhh,callfriend-eng_n-4175-M2,callfriend-eng_n-4175-M1,no Janice was talking to one of her friends an...,1,11


In [25]:
data.shape

(4650829, 24)

## Save outputs for server operations.

In [26]:
data.to_csv(outputs_path, index=False, encoding='utf-8')